# WLTP Class 3 Traction Inverter → Operating Points

This notebook converts the standardised **WLTP Class 3** driving cycle into
electrical operating points for a traction inverter and summarises them as
weighted histograms suitable for semiconductor loss evaluation.

**Workflow**
1. Load the WLTP Class 3 speed profile (1 Hz, ~1800 s).
2. Apply a vehicle and motor model to derive mechanical power and DC-link current.
3. Visualise speed, mechanical power, and DC current over the drive cycle.
4. Build a 1-D histogram over DC current (10 bins) for loss-map lookup.
5. Build a 2-D histogram over (V, I) for full operating-point coverage.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pyplecs.mission_profile import (
    wltp_to_electrical, load_wltp, mission_profile_to_histogram,
    VehicleParams, MotorParams,
)

In [ ]:
wltp = load_wltp(3)
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(wltp["time_s"], wltp["speed_kmh"])
ax.set_xlabel("Time [s]"); ax.set_ylabel("Speed [km/h]")
ax.set_title("WLTP Class 3 Drive Cycle"); ax.grid(True)
plt.tight_layout(); plt.show()

In [ ]:
elec = wltp_to_electrical(wltp_class=3)
print(f"Samples: {len(elec)}")
print(f"V_dc: {elec['V'].iloc[0]:.0f} V (constant)")
print(f"I range: {elec['I'].min():.1f} – {elec['I'].max():.1f} A")
print(f"P_mech range: {elec['P_mech'].min():.0f} – {elec['P_mech'].max():.0f} W")

In [ ]:
fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
ax1.plot(elec["time_s"], elec["speed_kmh"]); ax1.set_ylabel("Speed [km/h]"); ax1.set_title("WLTP Class 3 → Electrical"); ax1.grid(True)
ax2.plot(elec["time_s"], elec["P_mech"]); ax2.set_ylabel("P_mech [W]"); ax2.grid(True)
ax3.plot(elec["time_s"], elec["I"]); ax3.set_ylabel("I_dc [A]"); ax3.set_xlabel("Time [s]"); ax3.grid(True)
plt.tight_layout(); plt.show()

In [ ]:
tbl_1d = mission_profile_to_histogram(elec, columns=["I"], n_bins=10)
print(tbl_1d.summary())
fig, ax = plt.subplots(figsize=(10, 5))
tbl_1d.plot(ax=ax, kind="bar")
plt.tight_layout(); plt.show()

In [ ]:
tbl_2d = mission_profile_to_histogram(elec, columns=["V", "I"], n_bins=[3, 8])
print(tbl_2d.summary())
fig, ax = plt.subplots(figsize=(10, 6))
tbl_2d.plot(ax=ax, kind="heatmap")
plt.tight_layout(); plt.show()